# BraTS 2021 Preprocessing ve Veri Pipeline Dogrulama

Bu notebook, Faz 2 veri pipeline'inin dogru calistigini gorsel ve sayisal olarak dogrular.

Kapsam:

- Ham BraTS vakasini yukleme
- Preprocessing adimlarini tek tek inceleme
- Label donusumu dogrulama: `{0,1,2,4} -> {0,1,2,3}`
- Augmentation orneklerini gorsellestirme
- DataLoader batch testi
- Islenmis veri uzerinde sinif dagilimi kontrolu
- Tek vaka preprocessing suresi olcumu

In [ ]:
from __future__ import annotations

import copy
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.augmentation import BraTSAugmentation
from src.data.dataset import get_dataloaders
from src.data.preprocessing import BraTSPreprocessor

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

CONFIG_PATH = PROJECT_ROOT / "configs" / "training_config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

RAW_DIR = PROJECT_ROOT / config["data"]["data_dir"]
PROCESSED_DIR = PROJECT_ROOT / config["data"]["processed_dir"]
config["data"]["data_dir"] = str(RAW_DIR)
config["data"]["processed_dir"] = str(PROCESSED_DIR)
CASE_ID = "BraTS2021_00000"
CASE_DIR = RAW_DIR / CASE_ID
PROCESSED_CASE = PROCESSED_DIR / f"{CASE_ID}.npz"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw case: {CASE_DIR}")
print(f"Processed case: {PROCESSED_CASE}")

## Yardimci fonksiyonlar

In [ ]:
def case_file(case_dir: Path, modality: str) -> Path:
    return case_dir / f"{case_dir.name}_{modality}.nii.gz"


def load_nifti_array(path: Path, dtype=np.float32) -> np.ndarray:
    return np.asanyarray(nib.load(str(path)).dataobj).astype(dtype, copy=False)


def tumor_center_slices(mask: np.ndarray) -> dict[str, int]:
    coords = np.argwhere(mask > 0)
    if coords.size == 0:
        return {"sagittal": mask.shape[0] // 2, "coronal": mask.shape[1] // 2, "axial": mask.shape[2] // 2}
    center = np.round(coords.mean(axis=0)).astype(int)
    return {"sagittal": int(center[0]), "coronal": int(center[1]), "axial": int(center[2])}


def normalize_slice(slice_2d: np.ndarray) -> np.ndarray:
    values = slice_2d.astype(np.float32)
    nonzero = values[values != 0]
    if nonzero.size == 0:
        return values
    lo, hi = np.percentile(nonzero, [1, 99])
    return np.clip((values - lo) / (hi - lo + 1e-8), 0, 1)


def overlay_segmentation(base: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    base_norm = normalize_slice(base)
    rgb = np.stack([base_norm, base_norm, base_norm], axis=-1)
    colors = {
        1: np.array([1.0, 0.0, 0.0]),
        2: np.array([0.0, 1.0, 0.0]),
        3: np.array([0.0, 0.0, 1.0]),
        4: np.array([0.0, 0.0, 1.0]),
    }
    for label, color in colors.items():
        label_mask = mask == label
        rgb[label_mask] = (1 - alpha) * rgb[label_mask] + alpha * color
    return rgb


def plot_modalities(volumes: dict[str, np.ndarray], mask: np.ndarray, title: str) -> None:
    z = tumor_center_slices(mask)["axial"]
    modalities = ["t1", "t1ce", "t2", "flair"]
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, modality in zip(axes[:4], modalities):
        ax.imshow(normalize_slice(volumes[modality][:, :, z]).T, cmap="gray", origin="lower")
        ax.set_title(f"{modality} z={z}")
        ax.axis("off")
    overlay = overlay_segmentation(volumes["flair"][:, :, z], mask[:, :, z])
    axes[4].imshow(overlay.transpose(1, 0, 2), origin="lower")
    axes[4].set_title("FLAIR + mask")
    axes[4].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def print_nonzero_stats(images: np.ndarray, names=("t1", "t1ce", "t2", "flair")) -> None:
    for channel, name in enumerate(names):
        values = images[channel][images[channel] != 0]
        print(name, "mean", round(float(values.mean()), 4), "std", round(float(values.std()), 4), "nonzero", values.size)

## 1. Ham veri yukleme

Bir BraTS vakasi ham haliyle yuklenir. Orijinal shape ve etiket seti kontrol edilir.

In [ ]:
raw_volumes = {modality: load_nifti_array(case_file(CASE_DIR, modality), dtype=np.float32) for modality in ["t1", "t1ce", "t2", "flair"]}
raw_mask = load_nifti_array(case_file(CASE_DIR, "seg"), dtype=np.int16)

print("Raw shapes:")
for modality, volume in raw_volumes.items():
    print(f"  {modality}: {volume.shape}, dtype={volume.dtype}")
print(f"  seg: {raw_mask.shape}, labels={np.unique(raw_mask).tolist()}")

plot_modalities(raw_volumes, raw_mask, f"Ham veri - {CASE_ID}")

## 2. Preprocessing adimlari

Ayni vaka uzerinde preprocessing adimlari tek tek calistirilir:

1. Beyin bounding box'ina kirpma
2. `128x128x128` boyutuna resize
3. Non-zero beyin voxel'leri uzerinden z-score normalizasyon
4. Label donusumu

In [ ]:
preprocessor = BraTSPreprocessor(config)
volumes_list = [raw_volumes[modality] for modality in ["t1", "t1ce", "t2", "flair"]]

cropped_volumes, cropped_mask = preprocessor.crop_to_nonzero(volumes_list, raw_mask)
print("Original shape:", raw_mask.shape)
print("Cropped shape:", cropped_mask.shape)

cropped_dict = dict(zip(["t1", "t1ce", "t2", "flair"], cropped_volumes))
plot_modalities(cropped_dict, cropped_mask, "Beyin bolgesine kirpilmis veri")

In [ ]:
resized_volumes = [preprocessor.resize_volume(volume, preprocessor.target_size, is_mask=False) for volume in cropped_volumes]
resized_mask = preprocessor.resize_volume(cropped_mask, preprocessor.target_size, is_mask=True)
resized_dict = dict(zip(["t1", "t1ce", "t2", "flair"], resized_volumes))

print("Resized image shape:", resized_volumes[0].shape)
print("Resized mask shape:", resized_mask.shape)
print("Resized raw labels:", np.unique(resized_mask).tolist())
plot_modalities(resized_dict, resized_mask, "128x128x128 resize sonrasi")

In [ ]:
normalized_volumes = [preprocessor.zscore_normalize(volume) for volume in resized_volumes]
images = np.stack(normalized_volumes, axis=0).astype(np.float32)
converted_mask = preprocessor.convert_labels(resized_mask)
normalized_dict = dict(zip(["t1", "t1ce", "t2", "flair"], normalized_volumes))

print("Final images shape:", images.shape, images.dtype)
print("Final mask shape:", converted_mask.shape, converted_mask.dtype)
print("Converted labels:", np.unique(converted_mask).tolist())
print_nonzero_stats(images)
plot_modalities(normalized_dict, converted_mask, "Normalize edilmis ve label donusturulmus veri")

In [ ]:
hist_rows = []
for channel, name in enumerate(["t1", "t1ce", "t2", "flair"]):
    values = images[channel][images[channel] != 0]
    sample = values[np.linspace(0, values.size - 1, min(values.size, 50_000)).astype(int)]
    hist_rows.append(pd.DataFrame({"modality": name, "value": sample}))

hist_df = pd.concat(hist_rows, ignore_index=True)
g = sns.FacetGrid(hist_df, col="modality", col_wrap=2, sharex=True, sharey=False, height=3)
g.map_dataframe(sns.histplot, x="value", bins=80, stat="density")
g.fig.suptitle("Z-score sonrasi yogunluk histogramlari", y=1.03)
plt.show()

## 3. Label donusumu dogrulamasi

BraTS 2021 maskelerinde label `3` yoktur. Model egitimi icin label `4`, ard???k sinif indeksine yani `3` degerine donusturulur.

In [ ]:
toy_mask = np.array([0, 1, 2, 4], dtype=np.int16)
converted = preprocessor.convert_labels(toy_mask)
restored = preprocessor.inverse_convert_labels(converted)
print("Raw labels:      ", toy_mask.tolist())
print("Train labels:    ", converted.tolist())
print("Restored labels: ", restored.tolist())
assert converted.tolist() == [0, 1, 2, 3]
assert restored.tolist() == [0, 1, 2, 4]
print("Label conversion OK")

## 4. Augmentation ornekleri

Ayni islenmis vaka uzerinde flip, rotation, intensity shift, gaussian noise ve elastic deformation ornekleri gosterilir. Geometrik donusumler image ve mask'a ayni parametrelerle uygulanir; intensity donusumleri yalnizca image kanallarina uygulanir.

In [ ]:
processed = np.load(PROCESSED_CASE)
processed_images = processed["images"].astype(np.float32)
processed_mask = processed["mask"].astype(np.int64)
aug = BraTSAugmentation(config, seed=42)

examples = []
examples.append(("Original", processed_images, processed_mask))
examples.append(("Flip depth", *aug.random_flip(processed_images, processed_mask, axis=0, p=1.0)))
examples.append(("Rotation", *BraTSAugmentation(config, seed=1).random_rotation_3d(processed_images, processed_mask, max_angle=10)))
examples.append(("Intensity shift", aug.random_intensity_shift(processed_images, shift_range=0.1), processed_mask))
examples.append(("Gaussian noise", aug.random_gaussian_noise(processed_images, sigma=0.03), processed_mask))
examples.append(("Elastic deformation", *BraTSAugmentation(config, seed=2).elastic_deformation_3d(processed_images, processed_mask, alpha=8, sigma=4)))

for name, aug_images, aug_mask in examples:
    assert aug_images.shape == processed_images.shape
    assert aug_mask.shape == processed_mask.shape
    assert set(np.unique(aug_mask)).issubset({0, 1, 2, 3})
print("Augmentation examples validated")

In [ ]:
z = tumor_center_slices(processed_mask)["axial"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, aug_images, aug_mask) in zip(axes.ravel(), examples):
    overlay = overlay_segmentation(aug_images[3, :, :, z], aug_mask[:, :, z])
    ax.imshow(overlay.transpose(1, 0, 2), origin="lower")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. DataLoader testi

Train DataLoader'dan bir batch alinir ve shape'ler dogrulanir.

In [ ]:
loader_config = copy.deepcopy(config)
# Notebook dogrulamasini deterministik ve hizli tutmak icin elastic'i DataLoader testinde kapatiyoruz.
loader_config["augmentation"]["elastic_deformation"]["enabled"] = False
loaders = get_dataloaders(loader_config)
train_images, train_masks = next(iter(loaders["train"]))
print("Train batch images:", train_images.shape, train_images.dtype)
print("Train batch masks:", train_masks.shape, train_masks.dtype)
assert tuple(train_images.shape) == (2, 4, 128, 128, 128)
assert tuple(train_masks.shape) == (2, 128, 128, 128)
print("DataLoader OK")

In [ ]:
test_images, test_masks, metadata = next(iter(loaders["test"]))
print("Test batch images:", test_images.shape)
print("Test batch masks:", test_masks.shape)
print("Metadata case_id:", metadata["case_id"][0])

## 6. Islenmis veride sinif dagilimi

Ilk 100 islenmis vaka uzerinde label dagilimi hesaplanir. Bu kisim class imbalance'i gormek icin pratik bir kontroldur.

In [ ]:
processed_files = sorted(PROCESSED_DIR.glob("*.npz"))[:100]
label_counts = {0: 0, 1: 0, 2: 0, 3: 0}
case_rows = []

for path in processed_files:
    mask = np.load(path)["mask"]
    labels, counts = np.unique(mask, return_counts=True)
    current = {int(label): int(count) for label, count in zip(labels, counts)}
    for label in label_counts:
        label_counts[label] += current.get(label, 0)
    tumor_voxels = sum(current.get(label, 0) for label in [1, 2, 3])
    case_rows.append({"case_id": path.stem, "tumor_voxels": tumor_voxels, "background_voxels": current.get(0, 0)})

label_df = pd.DataFrame({"label": list(label_counts.keys()), "voxels": list(label_counts.values())})
label_df["percent"] = label_df["voxels"] / label_df["voxels"].sum() * 100
case_df = pd.DataFrame(case_rows)
print(f"Analysed processed cases: {len(processed_files)}")
display(label_df)
print("Background / tumor ratio summary")
display((case_df["background_voxels"] / case_df["tumor_voxels"].clip(lower=1)).describe().round(2).to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=label_df, x="label", y="percent", ax=axes[0])
axes[0].set_title("Label yuzdesi - ilk 100 processed vaka")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Voxel yuzdesi")

sns.histplot(case_df["tumor_voxels"], bins=30, ax=axes[1])
axes[1].set_title("Tumor voxel dagilimi - ilk 100 vaka")
axes[1].set_xlabel("Tumor voxel count")
plt.tight_layout()
plt.show()

## 7. Tek vaka preprocessing suresi

In [ ]:
start = time.perf_counter()
timed_images, timed_mask = preprocessor.preprocess_case(CASE_DIR)
elapsed = time.perf_counter() - start
print(f"Single case preprocessing time: {elapsed:.2f} seconds")
print(timed_images.shape, timed_mask.shape, np.unique(timed_mask).tolist())

## Sonuc

Bu notebook basariyla calistiginda Faz 2 veri pipeline'i icin su noktalar dogrulanmis olur:

- Ham NIfTI dosyalari okunabiliyor.
- Preprocessing sonucu `images=(4,128,128,128)` ve `mask=(128,128,128)` uretiliyor.
- Label donusumu model egitimi icin `{0,1,2,3}` formatina geciyor.
- Augmentation sonrasinda shape ve mask label'lari korunuyor.
- DataLoader batch uretiyor ve test loader vaka metadata'si donduruyor.
- Islenmis veri uzerindeki class imbalance gozlemlenebiliyor.